# Compare Model Outputs

Compares model outputs across runs using the composite objective score. Lower is better.

If `objective_score` is missing, the notebook falls back to time columns (`total_time` or `best_time`).

In [ ]:
import csv
from pathlib import Path

def _resolve_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return paths[0]

OUT = _resolve_existing([Path("outputs"), Path("../outputs")])

FILES = {
    "exact_mip": OUT / "exact_mip_all_runs_summary.csv",
    "baseline_greedy": OUT / "baseline_greedy_all_runs_summary.csv",
    "baseline_beam": OUT / "baseline_beam_all_runs_summary.csv",
    "baseline_random": OUT / "baseline_random_all_runs_summary.csv",
}

def _read_times(path, candidates):
    if not path.exists():
        return {}
    with path.open("r", newline="") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return {}
    field = next((c for c in candidates if c in rows[0]), None)
    if field is None:
        return {}
    out = {}
    for r in rows:
        run = r.get("run_name", "canonical")
        try:
            out[run] = float(r[field])
        except (TypeError, ValueError):
            continue
    return out

times = {
    "exact_mip": _read_times(FILES["exact_mip"], ["objective_score", "total_time"]),
    "baseline_greedy": _read_times(FILES["baseline_greedy"], ["objective_score", "total_time"]),
    "baseline_beam": _read_times(FILES["baseline_beam"], ["objective_score", "total_time"]),
    "baseline_random": _read_times(FILES["baseline_random"], ["objective_score", "total_time", "best_time"]),
}

all_runs = sorted(set().union(*[set(v.keys()) for v in times.values()]))
if not all_runs:
    raise RuntimeError("No model summary files found in outputs/. Run model notebooks first.")

comparison_rows = []
wins = {k: 0 for k in times.keys()}
for run in all_runs:
    row = {"run_name": run}
    best_model = None
    best_val = None
    for model in ["exact_mip", "baseline_greedy", "baseline_beam", "baseline_random"]:
        v = times[model].get(run)
        row[model] = "" if v is None else v
        if v is not None and (best_val is None or v < best_val):
            best_val = v
            best_model = model
    row["best_model"] = best_model if best_model is not None else ""
    row["best_objective_score"] = "" if best_val is None else best_val
    if best_model is not None:
        wins[best_model] += 1
    comparison_rows.append(row)

comparison_path = OUT / "model_output_comparison.csv"
with comparison_path.open("w", newline="") as f:
    w = csv.DictWriter(
        f,
        fieldnames=[
            "run_name",
            "exact_mip",
            "baseline_greedy",
            "baseline_beam",
            "baseline_random",
            "best_model",
            "best_objective_score",
        ],
    )
    w.writeheader()
    w.writerows(comparison_rows)

summary_rows = []
for model in ["exact_mip", "baseline_greedy", "baseline_beam", "baseline_random"]:
    vals = [times[model][r] for r in all_runs if r in times[model]]
    avg = (sum(vals) / len(vals)) if vals else ""
    summary_rows.append({
        "model": model,
        "n_runs": len(vals),
        "mean_objective_score": avg,
        "win_count": wins[model],
    })

summary_path = OUT / "model_output_comparison_summary.csv"
with summary_path.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["model", "n_runs", "mean_objective_score", "win_count"])
    w.writeheader()
    w.writerows(summary_rows)

print(f"Wrote: {comparison_path}")
print(f"Wrote: {summary_path}")
print("\nMean objective score by model (lower is better):")
for r in sorted(summary_rows, key=lambda x: float("inf") if x["mean_objective_score"] == "" else x["mean_objective_score"]):
    print(r)


Wrote: ../outputs/model_output_comparison.csv
Wrote: ../outputs/model_output_comparison_summary.csv

Mean objective time by model (lower is better):
{'model': 'exact_mip', 'n_runs': 500, 'mean_objective_time': 139.025, 'win_count': 500}
{'model': 'baseline_random', 'n_runs': 500, 'mean_objective_time': 139.43, 'win_count': 0}
{'model': 'baseline_beam', 'n_runs': 500, 'mean_objective_time': 140.846, 'win_count': 0}
{'model': 'baseline_greedy', 'n_runs': 500, 'mean_objective_time': 141.1835, 'win_count': 0}
